In [1]:
import GEOparse

In [ ]:
## some genes have multiple probes: they are taken average of . 
## some genes have no probes: they are set to -1
## this GPL96 platform has a column abs_call, which is supposed to be a classification of expressed, marginal or not expressed. That could be used instead, but not sure if it is common across platforms. 
## some platforms have multiple probes for the same gene. they are averaged.
## some platforms have no probes for some genes. they are set to -1. LOOK OUT FOR THIS IN DATA 
## check the PLATFORM_BLACKLIST and manually check the platforms in it to see if they can be fixed.

## KNOWN BLACKLISTS
#GPL10558 platform will be blacklisted as it is mostly empty and does not have the sane format as the others


Nomencalture: 
- GSE is a series set of data
- GSM is a single sample. It usually has a list of probes and corresponding expresson value
- GPL is a platform. Its related to how the expression analysis was carried out. each platform has probe-gene mappings. 
    - some genes might not have any probes in some platforms, some genes might have multiple

Things to do: 
- I've not added a disease state column as it is not a readily available field. Can maybe pass each gsm's metadata to an ai model's api and extract the information that way. 
    - Each GSE mentions the diseased or not in different ways, in different fields of metadata (title, charecteristics, etc)
    - for now, I have the title as a field in the o/p dataset as a placeholder
- Each sample seems to have been preprocessed, but different GSEs might have normalised the expression data differently. Need to look into some common standardisation scheme between the different GSEs

In [ ]:
## Add more gse_ids
gse_ids = [
    "GSE10325",
    "GSE38351",
    "GSE39088",
    "GSE49454",
    "GSE32591",
    "GSE45291", 
    "GSE72747"
] #from the https://doi.org/10.1371/journal.pone.0208132 paper
## 50722 removed as it is creating errors for some reason.

#GPL10558 platform will be blacklisted as it is mostly empty and does not have the sane format as the others

genes_list = [
    "MTHFR", "TYMS", "MTR", "MTRR",
    "CYP1A1", "GSTM1", "GSTT1", "GSTP1",
    "TLR2", "TLR4", "TLR7", "TLR9",
    "STAT4"
] #from the rupashree et al paper

In [ ]:
def process_platform(platform_id, genes_list):

    if platform_id in PLATFORM_BLACKLIST:
        print(f"Skipping platform {platform_id} as it is in the blacklist")
        return None

    gpl = GEOparse.get_GEO(platform_id)

    # Get annotation table
    annot = gpl.table

    # Filter
    result = annot[annot["Gene Symbol"].isin(genes_list)]

    #create a dictionary mapping gene symbols to their corresponding probe IDs
    gene_to_probe = {}

    #initialise to empty list for each gene symbol
    for gene in genes_list:
        gene_to_probe[gene] = []

    for _, row in result.iterrows():
        gene_symbol = row["Gene Symbol"]
        probe_id = row["ID"]
        if gene_symbol not in gene_to_probe:
            gene_to_probe[gene_symbol] = []
        gene_to_probe[gene_symbol].append(probe_id)
    
    return gene_to_probe

def summarise_probes(gene_to_probe):
    print("Summarising probes for each gene, platform:", platform_id)
    for gene, probes in gene_to_probe.items():
        print(f"{gene}: {len(probes)} probes")
    

In [40]:
def process_gsm(gsm, genes_list):
    """
    gsm is the sample object from GEOparse, 
    genes_list is the list of genes we are interested in
    """
    platform_id = gsm.metadata["platform_id"][0]

    if platform_id in PLATFORM_BLACKLIST:
        print(f"Skipping sample {gsm.name} in series {gsm.metadata['series_id']} with platform {platform_id} as it is in the blacklist")
        return None

    if len(gsm.metadata["platform_id"]) > 1:
        print(f"ERROR\nFound multiple platform IDs for sample {gsm.name} in series {gsm.metadata['series_id']}: {gsm.metadata['platform_id']}")

    
    #check if the platform has been processed before, if not process it and store the result in PROBES
    if platform_id not in PROBES and platform_id not in PLATFORM_BLACKLIST:
        gene_to_probe = process_platform(platform_id, genes_list)
        PROBES[platform_id] = gene_to_probe
    else:
        gene_to_probe = PROBES[platform_id]

    table = gsm.table
    gsm_filtered = table[table["ID_REF"].isin([probe for probes in gene_to_probe.values() for probe in probes])]

    gene_expression = {}
    for gene in genes_list:
        gene_ids = gene_to_probe[gene]
        gene_values = gsm_filtered[gsm_filtered["ID_REF"].isin(gene_ids)]["VALUE"].tolist()
        if len(gene_values) > 0:
            gene_expression[gene] = sum(gene_values) / len(gene_values)
        else: # if there are no values for the gene, set it to None
            gene_expression[gene] = -1
            print(f"WARNING: \nNo values found for gene {gene} in sample {gsm.name} in series {gsm.metadata['series_id']} with platform {platform_id}")
    return gene_expression
    


    

columns: GSE_ID, GSM_ID, GPL, title, [genes]

In [35]:
def process_gses(gse_ids, genes_list, filename = "database.csv"):
    
    file = open(filename, "w")
    #columns: GSE_ID, GSM_ID, GPL, title, [genes]
    file.write("GSE_ID,GSM_ID,GPL,title," + ",".join(genes_list) + "\n")
    
    for gse_id in gse_ids:
        gse = GEOparse.get_GEO(gse_id)
        samples = gse.gsms.keys()
        for sample in samples:
            gsm = gse.gsms[sample]
            gene_expression = process_gsm(gsm, genes_list)
            if gene_expression is None:
                # Skipping this sample as its platform is in the blacklist
                continue
            #print(f"Gene expression for sample {sample} in series {gse_id}: {gene_expression}")
            file.write(f"{gse_id},{sample},{gsm.metadata['platform_id'][0]},{gsm.metadata['title'][0]}," + ",".join([str(gene_expression[gene]) for gene in genes_list]) + "\n")
    file.close()

In [9]:
def get_platforms(list):
    ret_list = []

    for gse_id in gse_ids:
        gse  = GEOparse.get_GEO(gse_id)
        print(f"Series: {gse_id}, platforms: {gse.gpls.keys()}")
        ret_list.append(gse.gpls.keys())
    
    return ret_list


In [ ]:
## This line also downloads the raw data if it itsnt available locally, so it can take some time.
list_of_platforms = get_platforms(gse_ids)

In [30]:
l = []
for i in list_of_platforms:
    for j in i:
        l.append(j)

list_of_platforms = list(set(l))

print("List of platforms:", list_of_platforms)

List of platforms: ['GPL570', 'GPL13158', 'GPL96', 'GPL14663', 'GPL10558']


In [33]:
PROBES = {}
PLATFORM_BLACKLIST = []
for platform in list_of_platforms:
    try:
        gene_to_probe = process_platform(platform, genes_list)
        PROBES[platform] = gene_to_probe
    except: 
        print(f"ERROR processing platform {platform}, adding to blacklist")
        PLATFORM_BLACKLIST.append(platform)
    

14-Apr-2026 18:59:15 DEBUG utils - Directory ./ already exists. Skipping.
14-Apr-2026 18:59:15 INFO GEOparse - File already exist: using local version.
14-Apr-2026 18:59:15 INFO GEOparse - Parsing ./GPL570.txt: 
14-Apr-2026 18:59:15 DEBUG GEOparse - PLATFORM: GPL570
/home/aldis/.local/lib/python3.10/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
14-Apr-2026 18:59:16 DEBUG utils - Directory ./ already exists. Skipping.
14-Apr-2026 18:59:16 INFO GEOparse - File already exist: using local version.
14-Apr-2026 18:59:16 INFO GEOparse - Parsing ./GPL13158.txt: 
14-Apr-2026 18:59:16 DEBUG GEOparse - PLATFORM: GPL13158
/home/aldis/.local/lib/python3.10/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep=

ERROR processing platform GPL10558, adding to blacklist


In [39]:
print(PLATFORM_BLACKLIST)

['GPL10558']


In [38]:
print(list_of_platforms)

['GPL570', 'GPL13158', 'GPL96', 'GPL14663', 'GPL10558']


In [41]:
process_gses(gse_ids, genes_list)

14-Apr-2026 19:06:55 DEBUG utils - Directory ./ already exists. Skipping.
14-Apr-2026 19:06:55 INFO GEOparse - File already exist: using local version.
14-Apr-2026 19:06:55 INFO GEOparse - Parsing ./GSE10325_family.soft.gz: 
14-Apr-2026 19:06:55 DEBUG GEOparse - DATABASE: GeoMiame
14-Apr-2026 19:06:55 DEBUG GEOparse - SERIES: GSE10325
14-Apr-2026 19:06:55 DEBUG GEOparse - PLATFORM: GPL96
14-Apr-2026 19:06:56 DEBUG GEOparse - SAMPLE: GSM260886
14-Apr-2026 19:06:56 DEBUG GEOparse - SAMPLE: GSM260887
14-Apr-2026 19:06:56 DEBUG GEOparse - SAMPLE: GSM260888
14-Apr-2026 19:06:56 DEBUG GEOparse - SAMPLE: GSM260889
14-Apr-2026 19:06:56 DEBUG GEOparse - SAMPLE: GSM260890
14-Apr-2026 19:06:56 DEBUG GEOparse - SAMPLE: GSM260891
14-Apr-2026 19:06:56 DEBUG GEOparse - SAMPLE: GSM260892
14-Apr-2026 19:06:56 DEBUG GEOparse - SAMPLE: GSM260893
14-Apr-2026 19:06:56 DEBUG GEOparse - SAMPLE: GSM260894
14-Apr-2026 19:06:56 DEBUG GEOparse - SAMPLE: GSM260895
14-Apr-2026 19:06:56 DEBUG GEOparse - SAMPLE: GSM

No values found for gene TLR9 in sample GSM260886 in series ['GSE10325'] with platform GPL96
No values found for gene TLR9 in sample GSM260887 in series ['GSE10325'] with platform GPL96
No values found for gene TLR9 in sample GSM260888 in series ['GSE10325'] with platform GPL96
No values found for gene TLR9 in sample GSM260889 in series ['GSE10325'] with platform GPL96
No values found for gene TLR9 in sample GSM260890 in series ['GSE10325'] with platform GPL96
No values found for gene TLR9 in sample GSM260891 in series ['GSE10325'] with platform GPL96
No values found for gene TLR9 in sample GSM260892 in series ['GSE10325'] with platform GPL96
No values found for gene TLR9 in sample GSM260893 in series ['GSE10325'] with platform GPL96
No values found for gene TLR9 in sample GSM260894 in series ['GSE10325'] with platform GPL96
No values found for gene TLR9 in sample GSM260895 in series ['GSE10325'] with platform GPL96
No values found for gene TLR9 in sample GSM260896 in series ['GSE10325

14-Apr-2026 19:06:57 DEBUG GEOparse - PLATFORM: GPL570
/home/aldis/.local/lib/python3.10/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940450
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940451
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940452
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940453
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940454
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940455
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940456
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940457
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940458
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940459
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940460
14-Apr-2026 19:06:58 DEBUG GEOparse - SAMPLE: GSM940461
14-Apr-2026 19:06:58 DEBUG GEOparse - S

No values found for gene TLR9 in sample GSM940450 in series ['GSE38351'] with platform GPL96
No values found for gene TLR9 in sample GSM940451 in series ['GSE38351'] with platform GPL96
No values found for gene TLR9 in sample GSM940452 in series ['GSE38351'] with platform GPL96
No values found for gene TLR9 in sample GSM940453 in series ['GSE38351'] with platform GPL96
No values found for gene TLR9 in sample GSM940454 in series ['GSE38351'] with platform GPL96
No values found for gene TLR9 in sample GSM940455 in series ['GSE38351'] with platform GPL96
No values found for gene TLR9 in sample GSM940456 in series ['GSE38351'] with platform GPL96
No values found for gene TLR9 in sample GSM940457 in series ['GSE38351'] with platform GPL96
No values found for gene TLR9 in sample GSM940458 in series ['GSE38351'] with platform GPL96
No values found for gene TLR9 in sample GSM940459 in series ['GSE38351'] with platform GPL96
No values found for gene TLR9 in sample GSM940460 in series ['GSE38351

/home/aldis/.local/lib/python3.10/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955680
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955681
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955682
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955683
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955684
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955685
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955686
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955687
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955688
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955689
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955690
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955691
14-Apr-2026 19:07:02 DEBUG GEOparse - SAMPLE: GSM955692
14-Apr-2026 19:07:02 DEBUG GEOparse - 

Skipping sample GSM1199352 in series ['GSE49454'] with platform GPL10558 as it is in the blacklist
Skipping sample GSM1199353 in series ['GSE49454'] with platform GPL10558 as it is in the blacklist
Skipping sample GSM1199354 in series ['GSE49454'] with platform GPL10558 as it is in the blacklist
Skipping sample GSM1199355 in series ['GSE49454'] with platform GPL10558 as it is in the blacklist
Skipping sample GSM1199356 in series ['GSE49454'] with platform GPL10558 as it is in the blacklist
Skipping sample GSM1199357 in series ['GSE49454'] with platform GPL10558 as it is in the blacklist
Skipping sample GSM1199358 in series ['GSE49454'] with platform GPL10558 as it is in the blacklist
Skipping sample GSM1199359 in series ['GSE49454'] with platform GPL10558 as it is in the blacklist
Skipping sample GSM1199360 in series ['GSE49454'] with platform GPL10558 as it is in the blacklist
Skipping sample GSM1199361 in series ['GSE49454'] with platform GPL10558 as it is in the blacklist
Skipping s

14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807862
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807863
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807864
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807865
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807866
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807867
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807868
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807869
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807870
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807871
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807872
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807873
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807874
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807875
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807876
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807877
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GSM807878
14-Apr-2026 19:07:15 DEBUG GEOparse - SAMPLE: GS

No values found for gene GSTM1 in sample GSM807842 in series ['GSE32591', 'GSE32592'] with platform GPL14663
No values found for gene TLR9 in sample GSM807842 in series ['GSE32591', 'GSE32592'] with platform GPL14663
No values found for gene GSTM1 in sample GSM807843 in series ['GSE32591', 'GSE32592'] with platform GPL14663
No values found for gene TLR9 in sample GSM807843 in series ['GSE32591', 'GSE32592'] with platform GPL14663
No values found for gene GSTM1 in sample GSM807844 in series ['GSE32591', 'GSE32592'] with platform GPL14663
No values found for gene TLR9 in sample GSM807844 in series ['GSE32591', 'GSE32592'] with platform GPL14663
No values found for gene GSTM1 in sample GSM807845 in series ['GSE32591', 'GSE32592'] with platform GPL14663
No values found for gene TLR9 in sample GSM807845 in series ['GSE32591', 'GSE32592'] with platform GPL14663
No values found for gene GSTM1 in sample GSM807846 in series ['GSE32591', 'GSE32592'] with platform GPL14663
No values found for gen

/home/aldis/.local/lib/python3.10/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
14-Apr-2026 19:07:16 DEBUG GEOparse - SAMPLE: GSM1100843
14-Apr-2026 19:07:16 DEBUG GEOparse - SAMPLE: GSM1100844
14-Apr-2026 19:07:16 DEBUG GEOparse - SAMPLE: GSM1100845
14-Apr-2026 19:07:16 DEBUG GEOparse - SAMPLE: GSM1100846
14-Apr-2026 19:07:16 DEBUG GEOparse - SAMPLE: GSM1100847
14-Apr-2026 19:07:16 DEBUG GEOparse - SAMPLE: GSM1100848
14-Apr-2026 19:07:16 DEBUG GEOparse - SAMPLE: GSM1100849
14-Apr-2026 19:07:16 DEBUG GEOparse - SAMPLE: GSM1100850
14-Apr-2026 19:07:16 DEBUG GEOparse - SAMPLE: GSM1100851
14-Apr-2026 19:07:16 DEBUG GEOparse - SAMPLE: GSM1100852
14-Apr-2026 19:07:16 DEBUG GEOparse - SAMPLE: GSM1100853
14-Apr-2026 19:07:17 DEBUG GEOparse - SAMPLE: GSM1100854
14-Apr-2026 19:07:17 DEBUG GEOparse - SAMPLE: GSM1100855
14-Apr-2026 19:07:17 DEBU